In [1]:
import pandas as pd

df=pd.read_csv("../data/processed/creditcard_eda_processed.csv")
df

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V24,V25,V26,V27,V28,Class,scaled_amount,scaled_time,PCA1,PCA2
0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,...,0.066928,0.128539,-0.189115,0.133558,-0.021053,0,0.244964,-1.996583,-1.571678,0.675572
1,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,...,-0.339846,0.167170,0.125895,-0.008983,0.014724,0,-0.342475,-1.996583,1.086213,0.282673
2,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,...,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,0,1.160686,-1.996562,-2.053411,-1.077634
3,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,...,-1.175575,0.647376,-0.221929,0.062723,0.061458,0,0.140534,-1.996562,-1.150107,0.427442
4,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,...,0.141267,-0.206010,0.502292,0.219422,0.215153,0,-0.073403,-1.996541,-1.143820,1.341999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,4.356170,...,-0.509348,1.436807,0.250034,0.943651,0.823731,0,-0.350151,1.641931,-9.657134,9.457475
284803,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,-0.975926,...,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,0,-0.254117,1.641952,-0.467720,0.602071
284804,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,-0.484782,...,0.640134,0.265745,-0.087371,0.004455,-0.026561,0,-0.081839,1.641974,1.998489,-1.216424
284805,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,-0.399126,...,0.123205,-0.569159,0.546668,0.108821,0.104533,0,-0.313249,1.641974,0.040759,0.604718


In [2]:
x = df.drop("Class", axis=1)
y = df["Class"]

print("x Shape:", x.shape)
print("y Shape:", y.shape)

x Shape: (284807, 32)
y Shape: (284807,)


In [3]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.20, random_state=42, stratify=y)

print("Train:", x_train.shape, y_train.shape)
print("Test:", x_test.shape, y_test.shape)

Train: (227845, 32) (227845,)
Test: (56962, 32) (56962,)


In [4]:
from sklearn.utils import resample
import pandas as pd

train_df = pd.concat([x_train, y_train], axis=1)

fraud = train_df[train_df.Class == 1]
non_fraud = train_df[train_df.Class == 0]

non_fraud_sampled = resample(non_fraud, replace=False, n_samples=len(fraud) * 4,
                             random_state=42)

undersampled_df = pd.concat([fraud, non_fraud_sampled])

x_train_under = undersampled_df.drop("Class", axis=1)
y_train_under = undersampled_df["Class"]

print("After Undersampling:", x_train_under.shape, y_train_under.shape)

After Undersampling: (1970, 32) (1970,)


In [6]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

x_train_smote, y_train_smote = smote.fit_resample(x_train, y_train)

print("After SMOTE:", x_train_smote.shape, y_train_smote.shape)

After SMOTE: (454902, 32) (454902,)


In [7]:
print("Original:", y_train.value_counts())
print("/nSMOTE:", pd.Series(y_train_smote).value_counts())
print("/nUndersampled:", y_train_under.value_counts())

Original: Class
0    227451
1       394
Name: count, dtype: int64
/nSMOTE: Class
0    227451
1    227451
Name: count, dtype: int64
/nUndersampled: Class
0    1576
1     394
Name: count, dtype: int64


In [8]:
datasets = {
    "original": (x_train, y_train),
    "smote": (x_train_smote, y_train_smote),
    "undersampled": (x_train_under, y_train_under)
}

datasets

{'original': (              V1        V2        V3        V4        V5        V6        V7  \
  265518  1.946747 -0.752526 -1.355130 -0.661630  1.502822  4.024933 -1.479661   
  180305  2.035149 -0.048880 -3.058693  0.247945  2.943487  3.298697 -0.002192   
  42664  -0.991920  0.603193  0.711976 -0.992425 -0.825838  1.956261 -2.212603   
  198723  2.285718 -1.500239 -0.747565 -1.668119 -1.394143 -0.350339 -1.427984   
  82325  -0.448747 -1.011440  0.115903 -3.454854  0.715771 -0.147490  0.504347   
  ...          ...       ...       ...       ...       ...       ...       ...   
  233802  1.993864 -0.516866 -0.620118  0.129845 -0.285128  0.395044 -0.822358   
  85418  -1.497933  0.657921  1.581568 -0.024286  0.584698  1.303031  0.609212   
  29062   1.069777  0.072105  0.496540  1.505318 -0.380277 -0.370243  0.100551   
  13766   1.280465  0.300586  0.333044  0.512720  0.065052 -0.145844 -0.145519   
  17677  -0.598120  0.775041  1.823394  0.312991 -0.096171 -0.391452  0.499351   
  
 

In [9]:
import joblib

joblib.dump(x_train, "../data/processed/x_train_original.pkl")
joblib.dump(y_train, "../data/processed/y_train_original.pkl")

joblib.dump(x_train_smote, "../data/processed/x_train_smote.pkl")
joblib.dump(y_train_smote, "../data/processed/y_train_smote.pkl")

joblib.dump(x_train_under, "../data/processed/x_train_under.pkl")
joblib.dump(y_train_under, "../data/processed/y_train_under.pkl")

joblib.dump(x_test, "../data/processed/x_test.pkl")
joblib.dump(y_test, "../data/processed/y_test.pkl")

print("Preprocessing Saved Successfully!")

Preprocessing Saved Successfully!


In [10]:
df.to_csv("../data/processed/creditcard_preprocessed.csv", index=False)